In [1]:
# --- Imports: the building blocks for this lab ---

import os                          # Read environment variables (like your API key) from the OS
from dotenv import load_dotenv     # Load secrets from a local .env file into the environment
from scraper import fetch_website_contents  # Course helper: downloads a URL and returns plain text
from IPython.display import Markdown, display  # Render formatted markdown inside the notebook
from openai import OpenAI          # Official client for calling OpenAI's chat API

In [2]:
# --- Step 1: Load and validate your OpenAI API key ---
# The .env file keeps secrets out of source code. override=True means .env values
# win over any key already set in your shell (useful when re-running cells).

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')  # Look up the key by name from the environment

# Sanity checks before we make any paid API calls — catches the most common setup mistakes
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove the spaces or tabs")
else:
    print("API key found and looks good so far!")
    



API key found and looks good so far!


## Let's make a quick call to a Frontier 
# model to get started, as a preview!

In [22]:
# --- Step 2: Your first chat message ---
# The OpenAI Chat API expects a LIST of message dicts, not a single string.
# Each message has a "role" (who is speaking) and "content" (what they said).
# Roles you'll use most: "system" (instructions), "user" (your input), "assistant" (model replies).

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]  # One-turn conversation: just the user speaking

messages  # Display the structure so you can see exactly what gets sent to the API

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [23]:
# --- Step 3: Call the model ---
# OpenAI() automatically reads OPENAI_API_KEY from the environment (loaded in cell 1).
# chat.completions.create sends your messages and returns a structured response object.

openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",   # Which model to use (smaller/cheaper for this quick test)
    messages=messages     # The conversation history we built above
)

# The API returns metadata + a list of "choices". We take the first choice's text content.
response.choices[0].message.content

'Hi there! Nice to meet you, and welcome—thanks for saying hi.\n\nI can help with a lot of things: answer questions, explain topics, write or edit text, plan projects or trips, help with coding, math, or study ideas, and just chat. What would you like to do today?\n\nA few quick ideas:\n- Learn something new: ask me to explain a topic in simple terms\n- Draft or polish: emails, messages, resumes, essays\n- Plan something: a trip, a study plan, a to-do list\n- Solve problems: math, coding, puzzles\n- Have a casual chat or tell me your interests and I’ll tailor the conversation\n\nTell me one thing you’re curious about or what you’d like to start with.'

## Ok onwards with our first project

In [6]:
# --- Step 4: Fetch raw website text ---
# Our scraper downloads the page HTML and strips it down to readable text.
# This string is what we'll eventually pass to the LLM as context (not the URL itself).

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)  # Preview the scraped content — notice nav/footer text is still included

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

## Types of prompt

In [7]:
# --- Prompt engineering: the SYSTEM prompt ---
# The system prompt sets the model's persona and rules — it applies to the whole conversation.
# Think of it as "how should the assistant behave?" (tone, format, what to ignore).
# Experiment: change "Respond in markdown" to "Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, bumorous summary, ignoring text 
that might be navigation related. Respond in markdown. Do not wrap
the markdown in a code block - respond just with the markdown.
"""

In [8]:
# --- Prompt engineering: the USER prompt (task + data) ---
# The user prompt describes WHAT to do. Later we'll append the actual website text
# to this prefix, so the model receives both the instruction and the raw content.

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

## Messages

In [10]:
# --- How messages work in practice ---
# A real conversation is an ordered list: system first (rules), then user (question).
# The model reads ALL messages together to decide how to respond.

messages = [
    {"role": "system", "content": "You are a helpful assistant"},  # Sets behavior
    {"role": "user", "content": "What is 2 + 2?"}                  # The actual question
]

response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

response.choices[0].message.content  # The assistant's answer as plain text



'2 + 2 = 4'

In [11]:
# --- Build the message list for any website ---
# This function packages our prompts + scraped text into the format the API expects.
# String concatenation (prefix + website) puts instructions and data in one user message.

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},                    # Persona & output rules
        {"role": "user", "content": user_prompt_prefix + website}        # Task + raw page text
    ]

In [12]:
# --- Preview what we'd send to the API for Ed's site ---
# Run this to inspect the full payload before spending tokens on a real call.
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, bumorous summary, ignoring text \nthat might be navigation related. Respond in markdown. Do not wrap\nthe markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Time to bring it together - the API for OpenAI is very simple!

In [13]:
# --- The full pipeline: URL → summary text ---
# Three steps chained together:
#   1. Scrape the URL into plain text
#   2. Build the system + user messages
#   3. Ask the model and return its reply

def summarize(url):
    website = fetch_website_contents(url)          # Step 1: get page content
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",                      # Slightly larger model for better summaries
        messages=messages_for(website)             # Step 2: wrap content in prompts
    )
    return response.choices[0].message.content   # Step 3: extract the assistant's text

In [14]:
summarize("https://edwarddonner.com")

'# Edward Donner’s Digital Playground\n\nMeet Ed, your friendly neighborhood code geek and LLM evangelist who’s way too enthusiastic about AI and electronic music (amateur level, so listen at your own risk). He’s the big brain behind some AI startups, including one that got snatched up in 2021, and he apparently loves talking about Large Language Models so much that his friends forced him to put his ramblings into Udemy courses—now boasting 600,000 students globally. \n\nIf you want to watch LLMs throw diplomatic shade at each other, check out his "Outsmart" arena. Oh, and he’s been dropping AI course updates regularly—think of it as your roadmap for becoming an AI wizard:\n\n- Feb 2026: AI Coder: Vibe Coder to Agentic Engineer  \n- Jan 2026: AI Builder with n8n – Create Agents and Voice Agents  \n- Sept 2025: AI Engineering MLOps Track – Deploy AI to Production  \n- May 2025: Course sequencing advice for AI learners\n\nBasically, if you love AI, want to outsmart bots with bots, or jus

In [15]:
# --- Render the summary as formatted markdown in the notebook ---
# summarize() returns a raw string; Markdown() tells Jupyter to render headers, bold, etc.

def display_summary(url):
    summary = summarize(url)       # Get the LLM's markdown-formatted response
    display(Markdown(summary))     # Render it nicely instead of showing plain text

In [16]:
display_summary("https://edwarddonner.com")


# Edward Donner's Website: Where Code Meets Ego (and LLMs Duel)

Welcome to Ed’s personal playground, where he’s basically the Elon Musk of large language models (LLMs)—minus the rockets, plus a surprising amount of electronic music that’s “very amateur.” Ed’s the brain behind Nebula.io and a bunch of AI startups, with a résumé that screams “I was once a big shot at JPMorgan.”

His true passion? Rambling about AI to anyone who will listen, which led him to create some Udemy courses that somehow managed to snag 600,000 students worldwide. Not bad for someone who could be coding instead of annoying friends, right?

### Announcements to Note:
- New AI Coder and AI Builder resources dropped recently (because what’s better than more AI stuff?).
- A cool-sounding “AI Engineering MLOps Track” launched for you production nerds.
- A handy guide on the best order to take his AI courses—because random clicking is for amateurs.

He also runs this cheeky arena called "Outsmart" where LLMs go head-to-head in battles of wits and deceit. If you’re into robots scheming against each other, it’s your kind of fun.

### Quick hits:
- Email: ed[at]edwarddonner[dot]com (no spambots, please)
- Socials: LinkedIn, Twitter, Facebook (because why not)

Basically, if you love AI, code, and the occasional impromptu lecture you didn’t ask for, Ed’s your guy. Otherwise, brace yourself for some serious AI nerd vibes.

## Let's try more websites

In [17]:
display_summary("https://cnn.com")

Ah, CNN’s digital clubhouse: a sprawling maze of news categories, from the usual global chaos (Ukraine-Russia war, Israel-Hamas conflict) to sports, fashion, and even your daily dose of “Life, But Better.” It’s basically a news buffet with every imaginable flavor, plus a side of tech support on ads for when your video refuses to load. No actual headlines here, just a reminder that this is where the world’s breaking and latest news hang out—when you’re ready to sign in and dive into the chaos of current events and endless topics. Bonus: they really want to know if that annoying ad you saw was *that* annoying.

In [18]:
display_summary("https://anthropic.com")

Ah, Anthropic—where AI meets safety goggles and ethics manuals. This site screams “We’re serious about AI but also kinda fun,” featuring their star AI lineup with poetic names like Mythos, Fable, and Haiku (because who doesn’t want a robot embodying literary flair?). They’re all about wrestling the big scary questions on AI safety, governance, and societal impact like the responsible nerds they are.

Newsflash: their Fable 5 model just got a shiny export control waiver and will be unleashed unto the world starting July 1. So, if you’re into a safer, more thoughtful AI, Anthropic’s trying to convince you they’ve got you covered—complete with developer docs, tutorials, and some corporate virtue signaling with things like Claude’s Constitution and responsible scaling policies. Basically, your AI with a conscience.

In [24]:
display_summary("https://mwperrinejr.github.io/")

Ah yes, Michael Perrine’s data science portfolio: a love letter to spreadsheets, cats, and the Kansas City Chiefs. After a decade in finance, he’s now proudly wielding his freshly minted data science chops—complete with deep learning dreams and synthetic Kaggle data projects.

His portfolio is basically a mixed bag of nerdy goodness: sales prediction models, fraud detection pipelines, and an app that probably flags your fake transactions (because who doesn’t want a digital snitch?). When not wrangling data, Michael’s busy being a devoted husband, cat dad, and occasional golf ghost.

No earth-shattering news here—just a steady march toward data science domination, one logistic regression at a time.